# Detectree2: Multi-threshold Tree Crown Detection (SIT orthomosaics)

This notebook processes all orthomosaics in `input/input_om_sit/` and exports **one GPKG per orthomosaic** with **one layer per confidence threshold** (multi-threshold store).

Key properties:
- **Idempotent**: if the multi-threshold crowns already exist in the output folder and pass verification, Detectree inference is skipped for that orthomosaic.
- **Visual checks**: generates quick plots and map overlays so you can confirm things look right.
- **Storage cleanup**: optionally deletes intermediate tiling/prediction artifacts after successful export to avoid disk bloat.

In [ ]:
# =========================
# Parameters (edit here)
# =========================
from __future__ import annotations

from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Find the repo root by walking up until we see expected folders."""
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'input').exists() and (p / 'output').exists():
            return p
    # Fallback: if output doesn't exist yet, accept 'input' only
    for p in [start, *start.parents]:
        if (p / 'input').exists():
            return p
    raise RuntimeError(f'Could not locate project root above {start}')

# Project / data locations
PROJECT_ROOT = find_project_root(Path.cwd())
OM_SUBDIR = Path('input') / 'input_om_sit'  # <--- requested: SIT orthomosaics
OM_GLOB = 'sit_om*.tif'  # change to '*.tif' if you intentionally want non-sit files too
MODEL_REL_PATH = Path('input') / 'detectree_models' / '250312_flexi.pth'

# Output locations (new folder under output/)
RUN_FOLDER_NAME = 'detectree_om_sit_multithreshold'  # change if you want a different run folder
OUTPUT_ROOT = PROJECT_ROOT / 'output'
RUN_DIR = OUTPUT_ROOT / RUN_FOLDER_NAME
CROWNS_DIR = RUN_DIR / 'crowns_multithreshold'
WORK_DIR = RUN_DIR / 'work_detectree'  # tiles + predictions live here; can be deleted at end

# Performance / device
DEVICE = 'cpu'  # 'cpu' on macOS is typical; GPU only if your env supports it
NUM_THREADS = 6

# Tiling params (Detectree2)
TILE_BUFFER = 20
TILE_WIDTH = 45
TILE_HEIGHT = 45
DTYPE_BOOL = True

# Crown cleaning params
FIXED_IOU = 0.7
SIMPLIFY_TOL = 0.3  # set 0 to disable geometry simplify

# Multi-threshold sweep (the “future-proof” output you said you want)
import numpy as np
THRESHOLDS = [float(round(x, 2)) for x in np.arange(0.15, 0.651, 0.05)]  # 0.15..0.65 step 0.05

# Idempotency behavior
SKIP_IF_CROWNS_EXIST_AND_VALID = True
REBUILD_IF_INVALID = True  # if an existing GPKG is incomplete/invalid, rebuild it
OVERWRITE_EXISTING_CROWNS = False  # only used when rebuilding

# Visual checks
SAMPLE_OM_STEM_FOR_VIZ = 'sit_om1'  # pick any stem present in input/input_om_sit
SAMPLE_CONF_FOR_OVERLAY = 0.55  # confidence used for raster overlay sanity check

# Optional run filters (keep default to run everything)
ONLY_OMS: list[str] | None = None  # e.g. ['sit_om1','sit_om2']
LIMIT_OMS: int | None = None  # e.g. 3 to run first 3 only

# Storage cleanup (after successful run)
CLEANUP_DELETE_WORK_DIR = True  # deletes tiles/predictions under WORK_DIR
CLEANUP_DRY_RUN = False  # if True, prints what would be deleted without deleting

print('Project root:', PROJECT_ROOT)
print('Run dir     :', RUN_DIR)
print('OM dir      :', PROJECT_ROOT / OM_SUBDIR)
print('Model       :', PROJECT_ROOT / MODEL_REL_PATH)

In [ ]:
# =========================
# Imports + environment setup
# =========================
import os
import json
import glob
import re
from datetime import datetime

import pandas as pd
import geopandas as gpd
import fiona
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

import torch
from detectree2.preprocessing.tiling import tile_data
from detectree2.models.train import setup_cfg
from detectree2.models.predict import predict_on_data
from detectree2.models.outputs import project_to_geojson, stitch_crowns, clean_crowns
from detectron2.engine import DefaultPredictor

# Thread controls (CPU inference)
os.environ['OMP_NUM_THREADS'] = str(NUM_THREADS)
os.environ['MKL_NUM_THREADS'] = str(NUM_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(NUM_THREADS)
os.environ['NUMEXPR_NUM_THREADS'] = str(NUM_THREADS)
torch.set_num_threads(NUM_THREADS)
try:
    torch.set_num_interop_threads(NUM_THREADS)
except Exception:
    pass

# PROJ / pyproj safety (helps with CRS-related errors when writing/reading)
import pyproj

def ensure_proj_data_dir() -> None:
    try:
        _ = pyproj.datadir.get_data_dir()
        return
    except Exception:
        pass

    candidates: list[str] = []
    env_proj = os.environ.get('PROJ_DATA') or os.environ.get('PROJ_LIB')
    if env_proj:
        candidates.append(env_proj)

    candidates.extend([
        os.path.expanduser('~/anaconda3/share/proj'),
        os.path.expanduser('~/miniconda3/share/proj'),
        '/opt/homebrew/share/proj',
        '/usr/local/share/proj',
    ])

    # Try conda base if available
    try:
        import subprocess
        result = subprocess.run(['conda', 'info', '--base'], capture_output=True, text=True)
        conda_base = result.stdout.strip()
        if conda_base:
            candidates.append(os.path.join(conda_base, 'share', 'proj'))
    except Exception:
        pass

    for cand in candidates:
        if cand and os.path.exists(cand):
            pyproj.datadir.set_data_dir(cand)
            os.environ['PROJ_DATA'] = cand
            os.environ['PROJ_LIB'] = cand
            print(f'Set PROJ data directory: {cand}')
            return

    raise RuntimeError('Valid PROJ data directory not found; set PROJ_DATA/PROJ_LIB manually.')

ensure_proj_data_dir()

# Basic path checks
om_dir = (PROJECT_ROOT / OM_SUBDIR).resolve()
model_path = (PROJECT_ROOT / MODEL_REL_PATH).resolve()
if not om_dir.exists():
    raise FileNotFoundError(f'Orthomosaic folder not found: {om_dir}')
if not model_path.exists():
    raise FileNotFoundError(f'Detectree model not found: {model_path}')

RUN_DIR.mkdir(parents=True, exist_ok=True)
CROWNS_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
print('Orthomosaics folder:', om_dir)
print('Crowns output folder:', CROWNS_DIR)
print('Work folder:', WORK_DIR)

In [ ]:
# =========================
# Helper functions
# =========================
OM_NUM_PAT = re.compile(r'sit_om(\d+)', re.IGNORECASE)

def om_tag_from_stem(stem: str) -> str:
    m = OM_NUM_PAT.search(stem)
    if m:
        return f'OM{int(m.group(1))}'
    return stem

def layer_name(conf: float) -> str:
    return f"conf_{str(round(float(conf), 2)).replace('.', 'p')}"

def clean_bad_geojson_names(geo_folder: Path) -> int:
    bad = list(geo_folder.glob('*_None.geojson'))
    for p in bad:
        try:
            p.unlink()
        except Exception as e:
            print(f'Warning: could not remove {p}: {e}')
    return len(bad)

def discover_orthomosaics() -> list[Path]:
    paths = sorted(om_dir.glob(OM_GLOB))
    if ONLY_OMS:
        allow = set(ONLY_OMS)
        paths = [p for p in paths if p.stem in allow]
    if LIMIT_OMS is not None:
        paths = paths[: int(LIMIT_OMS)]
    if not paths:
        raise FileNotFoundError(f'No orthomosaics found in {om_dir} with pattern {OM_GLOB}')
    return paths

def setup_detection_config(trained_model_path: Path, device: str = 'cpu'):
    cfg = setup_cfg(update_model=str(trained_model_path))
    try:
        cfg.MODEL.DEVICE = device
    except Exception:
        pass
    return cfg

def ensure_predictions_geo(om_path: Path) -> Path:
    """Ensure Detectree predictions exist for this orthomosaic; returns predictions_geo folder."""
    stem = om_path.stem
    tiles_path = (WORK_DIR / stem).resolve()
    geo_folder = tiles_path / 'predictions_geo'

    if geo_folder.exists() and list(geo_folder.glob('*.geojson')):
        return geo_folder

    tiles_path.mkdir(parents=True, exist_ok=True)
    print(f'  - Tiling: {stem}')
    tile_data(
        str(om_path),
        str(tiles_path),
        TILE_BUFFER,
        TILE_WIDTH,
        TILE_HEIGHT,
        dtype_bool=DTYPE_BOOL,
    )

    cfg = setup_detection_config(model_path, device=DEVICE)
    predictor = DefaultPredictor(cfg)
    print(f'  - Predicting: {stem}')
    predict_on_data(directory=str(tiles_path), predictor=predictor)

    predictions_folder = tiles_path / 'predictions'
    print(f'  - Projecting to geojson: {stem}')
    project_to_geojson(str(tiles_path), str(predictions_folder), str(geo_folder))

    if not (geo_folder.exists() and list(geo_folder.glob('*.geojson'))):
        raise RuntimeError(f'No geojson predictions produced for {stem} in {geo_folder}')
    return geo_folder

def stitch_base_crowns(geo_folder: Path) -> gpd.GeoDataFrame:
    removed = clean_bad_geojson_names(geo_folder)
    if removed:
        print(f'  - Removed {removed} invalid geojson files in {geo_folder}')
    base = stitch_crowns(str(geo_folder), 1)
    base = base[base.is_valid]
    return base

def build_multithreshold_store(
    stem: str,
    base: gpd.GeoDataFrame,
    gpkg_path: Path,
    meta_path: Path,
    thresholds: list[float],
    fixed_iou: float,
    simplify_tol: float,
    overwrite: bool,
    ) -> dict:
    if overwrite and gpkg_path.exists():
        gpkg_path.unlink()

    layer_counts: dict[str, int] = {}
    for conf in thresholds:
        g = base.copy()
        if simplify_tol and simplify_tol > 0:
            g = g.set_geometry(g.geometry.simplify(simplify_tol))
        g = clean_crowns(g, fixed_iou, float(conf))
        g = g[g.is_valid]
        lyr = layer_name(conf)
        g['orthomosaic'] = stem
        g['confidence'] = float(conf)
        g['threshold_tag'] = lyr
        g.to_file(str(gpkg_path), layer=lyr, driver='GPKG')
        layer_counts[lyr] = int(len(g))

    meta = {
        'orthomosaic': stem,
        'gpkg_path': str(gpkg_path),
        'thresholds': [float(x) for x in thresholds],
        'layer_counts': layer_counts,
        'created_at': datetime.utcnow().isoformat() + 'Z',
    }
    meta_path.write_text(json.dumps(meta, indent=2))
    return meta

def validate_multithreshold_store(
    gpkg_path: Path,
    meta_path: Path,
    stem: str,
    thresholds: list[float],
    strict: bool = True,
    ) -> dict:
    if not gpkg_path.exists():
        return {'ok': False, 'reason': 'gpkg_missing'}
    if not meta_path.exists():
        return {'ok': False, 'reason': 'meta_missing'}

    expected_layers = [layer_name(c) for c in thresholds]
    layers = list(fiona.listlayers(str(gpkg_path)))
    missing = sorted(set(expected_layers) - set(layers))
    if missing:
        return {'ok': False, 'reason': 'missing_layers', 'missing_layers': missing}

    # Lightweight attribute/value check (read a tiny sample per layer)
    for conf in thresholds:
        lyr = layer_name(conf)
        g = gpd.read_file(str(gpkg_path), layer=lyr)
        for col in ['orthomosaic', 'confidence', 'threshold_tag']:
            if col not in g.columns:
                return {'ok': False, 'reason': 'missing_columns', 'layer': lyr, 'missing_column': col}
        if strict and len(g) > 0:
            if not (g['orthomosaic'].astype(str) == stem).all():
                return {'ok': False, 'reason': 'bad_orthomosaic_values', 'layer': lyr}
            if not (g['threshold_tag'].astype(str) == lyr).all():
                return {'ok': False, 'reason': 'bad_threshold_tag_values', 'layer': lyr}
            if not np.isclose(g['confidence'].astype(float), float(conf)).all():
                return {'ok': False, 'reason': 'bad_confidence_values', 'layer': lyr}

    # Meta sanity
    try:
        meta = json.loads(meta_path.read_text())
    except Exception as e:
        return {'ok': False, 'reason': f'meta_read_failed: {e}'}
    if str(gpkg_path) != meta.get('gpkg_path'):
        # allow path changes if moved; not fatal
        pass
    return {'ok': True, 'reason': 'ok', 'layers': len(expected_layers)}

In [ ]:
# =========================
# Run: build / verify multi-threshold crown stores
# =========================
om_paths = discover_orthomosaics()
print(f'Found {len(om_paths)} orthomosaics to process.')

run_rows: list[dict] = []

for om_path in om_paths:
    stem = om_path.stem
    tag = om_tag_from_stem(stem)
    gpkg_path = CROWNS_DIR / f'{tag}_multithreshold.gpkg'
    meta_path = CROWNS_DIR / f'{tag}_multithreshold.meta.json'

    print(f'\n=== {stem} ({tag}) ===')
    print('Orthomosaic:', om_path)
    print('Output gpkg:', gpkg_path)

    if SKIP_IF_CROWNS_EXIST_AND_VALID:
        v = validate_multithreshold_store(
            gpkg_path=gpkg_path,
            meta_path=meta_path,
            stem=stem,
            thresholds=THRESHOLDS,
            strict=True,
        )
        if v.get('ok'):
            print('  - Crowns already exist and are valid → skipping Detectree + export')
            # Collect counts (cheap: from meta)
            meta = json.loads(meta_path.read_text())
            run_rows.append({
                'orthomosaic': stem,
                'tag': tag,
                'status': 'skipped_existing',
                'gpkg': str(gpkg_path),
                'meta': str(meta_path),
                'total_crowns_all_layers': int(sum(meta.get('layer_counts', {}).values())),
            })
            continue
        else:
            print(f"  - Existing crowns not usable ({v.get('reason')}); will rebuild")
            if not (REBUILD_IF_INVALID or OVERWRITE_EXISTING_CROWNS):
                raise RuntimeError(f'Existing crowns invalid for {stem} and rebuild disabled: {v}')

    # Ensure predictions exist (runs Detectree if needed)
    geo_folder = ensure_predictions_geo(om_path)
    base = stitch_base_crowns(geo_folder)
    if base is None or len(base) == 0:
        raise RuntimeError(f'No stitched crowns for {stem} (geo_folder={geo_folder})')

    # Export multi-threshold store
    meta = build_multithreshold_store(
        stem=stem,
        base=base,
        gpkg_path=gpkg_path,
        meta_path=meta_path,
        thresholds=THRESHOLDS,
        fixed_iou=FIXED_IOU,
        simplify_tol=SIMPLIFY_TOL,
        overwrite=bool(OVERWRITE_EXISTING_CROWNS or REBUILD_IF_INVALID),
    )

    # Verify
    v2 = validate_multithreshold_store(
        gpkg_path=gpkg_path,
        meta_path=meta_path,
        stem=stem,
        thresholds=THRESHOLDS,
        strict=True,
    )
    if not v2.get('ok'):
        raise AssertionError(f'Verification failed for {stem}: {v2}')

    run_rows.append({
        'orthomosaic': stem,
        'tag': tag,
        'status': 'built',
        'gpkg': str(gpkg_path),
        'meta': str(meta_path),
        'total_crowns_all_layers': int(sum(meta.get('layer_counts', {}).values())),
    })
    print('  - Done')

run_df = pd.DataFrame(run_rows).sort_values(['tag', 'orthomosaic']).reset_index(drop=True)
display(run_df)

# Save run summary inside the new output folder
summary_csv = RUN_DIR / 'run_summary.csv'
summary_json = RUN_DIR / 'run_summary.json'
run_df.to_csv(summary_csv, index=False)
summary_json.write_text(run_df.to_json(orient='records', indent=2))
print('Wrote:', summary_csv)
print('Wrote:', summary_json)
print('Outputs:', CROWNS_DIR)

In [ ]:
# =========================
# Visual sanity checks
# =========================
# (1) Per-OM counts vs confidence (from meta, fast)
meta_rows = []
for r in run_rows:
    mp = Path(r['meta'])
    if not mp.exists():
        continue
    m = json.loads(mp.read_text())
    stem = m.get('orthomosaic')
    counts = m.get('layer_counts', {})
    for conf in THRESHOLDS:
        lyr = layer_name(conf)
        meta_rows.append({
            'orthomosaic': stem,
            'confidence': float(conf),
            'count': int(counts.get(lyr, 0)),
        })

counts_all_df = pd.DataFrame(meta_rows)
if counts_all_df.empty:
    raise RuntimeError('No meta-derived counts found for visual checks.')

plt.figure(figsize=(7, 3.5))
for om_name, sub in counts_all_df.groupby('orthomosaic'):
    plt.plot(sub['confidence'], sub['count'], alpha=0.25, linewidth=1)
med = counts_all_df.groupby('confidence')['count'].median().reset_index()
plt.plot(med['confidence'], med['count'], color='black', linewidth=2, label='median')
plt.xlabel('Confidence threshold')
plt.ylabel('Crown count')
plt.title('Crown counts across orthomosaics (per threshold)')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

# (2) Boxplot: distribution of counts per threshold
plt.figure(figsize=(10, 3.5))
order = sorted(counts_all_df['confidence'].unique())
data = [counts_all_df.loc[counts_all_df['confidence'] == c, 'count'].values for c in order]
plt.boxplot(data, labels=[f'{c:.2f}' for c in order], showfliers=False)
plt.xlabel('Confidence threshold')
plt.ylabel('Crown count')
plt.title('Distribution of crown counts per threshold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# (3) Sample OM: map grid across thresholds
sample_stem = SAMPLE_OM_STEM_FOR_VIZ
if sample_stem not in set(run_df['orthomosaic']):
    # fallback to first processed OM
    sample_stem = str(run_df.iloc[0]['orthomosaic'])
sample_tag = om_tag_from_stem(sample_stem)
sample_gpkg = CROWNS_DIR / f'{sample_tag}_multithreshold.gpkg'
if not sample_gpkg.exists():
    raise FileNotFoundError(f'Sample GPKG not found: {sample_gpkg}')

viz_thresholds = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65]
viz_thresholds = [c for c in viz_thresholds if c in set(THRESHOLDS)]
viz_layers = [layer_name(c) for c in viz_thresholds]

n = len(viz_layers)
cols = 3
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
axes = np.array(axes).reshape(-1)
for ax in axes[n:]:
    ax.axis('off')

for ax, conf, lyr in zip(axes, viz_thresholds, viz_layers):
    g = gpd.read_file(str(sample_gpkg), layer=lyr)
    if len(g) == 0:
        ax.set_title(f'{sample_stem} conf {conf:.2f} (0)')
        ax.axis('off')
        continue
    g.plot(ax=ax, color='#2ca02c', alpha=0.45, edgecolor='black', linewidth=0.2)
    ax.set_title(f'{sample_stem} conf {conf:.2f} ({len(g)})')
    ax.set_axis_off()
plt.tight_layout()
plt.show()

# (4) Overlay crowns on the orthomosaic raster (one threshold)
overlay_conf = float(SAMPLE_CONF_FOR_OVERLAY)
overlay_layer = layer_name(overlay_conf)
overlay_gdf = gpd.read_file(str(sample_gpkg), layer=overlay_layer)

sample_tif = om_dir / f'{sample_stem}.tif'
if sample_tif.exists():
    fig, ax = plt.subplots(figsize=(10, 10))
    with rasterio.open(sample_tif) as src:
        show(src, ax=ax)
    if len(overlay_gdf) > 0:
        overlay_gdf.plot(ax=ax, color='#2ca02c', alpha=0.35, edgecolor='black', linewidth=0.2)
    ax.set_title(f'{sample_stem}: crowns overlay (conf {overlay_conf:.2f})')
    ax.set_axis_off()
    plt.show()
else:
    print('Skipping raster overlay; orthomosaic not found:', sample_tif)

## Output format (multi-threshold store)

This run writes everything under:
- `output/{RUN_FOLDER_NAME}/`

Per orthomosaic:
- `crowns_multithreshold/OM{n}_multithreshold.gpkg` — one layer per confidence threshold (`conf_0p15`, …).
- `crowns_multithreshold/OM{n}_multithreshold.meta.json` — sidecar metadata with layer counts.

Idempotency:
- If a GPKG + meta already exist and pass verification, that orthomosaic is **skipped** (no Detectree inference).

Intermediate data:
- `work_detectree/` stores tiles + predictions; set `CLEANUP_DELETE_WORK_DIR=True` to delete it at the end.

In [ ]:
# =========================
# Cleanup: remove large intermediate artifacts
# =========================
import shutil

if not CLEANUP_DELETE_WORK_DIR:
    print('Cleanup disabled (CLEANUP_DELETE_WORK_DIR=False).')
else:
    # Safety: only delete work dir if all outputs validate
    failures = []
    for _, r in run_df.iterrows():
        stem = str(r['orthomosaic'])
        tag = str(r['tag'])
        gpkg_path = CROWNS_DIR / f'{tag}_multithreshold.gpkg'
        meta_path = CROWNS_DIR / f'{tag}_multithreshold.meta.json'
        v = validate_multithreshold_store(
            gpkg_path=gpkg_path,
            meta_path=meta_path,
            stem=stem,
            thresholds=THRESHOLDS,
            strict=True,
        )
        if not v.get('ok'):
            failures.append((stem, v))

    if failures:
        print('Not cleaning up: one or more outputs failed validation:')
        for stem, v in failures:
            print('  -', stem, v)
        raise RuntimeError('Cleanup aborted due to failed validation.')

    if not WORK_DIR.exists():
        print('Work dir already gone:', WORK_DIR)
    else:
        if CLEANUP_DRY_RUN:
            print('[DRY RUN] Would delete work dir:', WORK_DIR)
        else:
            print('Deleting work dir (tiles + predictions):', WORK_DIR)
            shutil.rmtree(WORK_DIR)
            print('Deleted:', WORK_DIR)